# Exercise 9: fetching data from an API

The KMI publishes its observations through an API: `https://opendata.meteo.be/service/ows`. Our files only cover 2023; the API has more.
What the datasets and their columns mean: https://opendata.meteo.be/documentation/?dataset=aws

In [1]:
import pandas as pd

KMI_URL = "https://opendata.meteo.be/service/ows"

**9.1** Fetch the list of KMI weather stations (dataset `aws:aws_station`) as JSON, and make a
DataFrame with the code, name and altitude of every station. What is the code of Uccle?

In [2]:
import requests

params = {
    "service": "WFS",
    "version": "2.0.0",
    "request": "GetFeature",
    "typeNames": "aws:aws_station",
    "outputFormat": "json",
}
response = requests.get(KMI_URL, params=params, timeout=30)
response.raise_for_status()
stations = pd.json_normalize(response.json()["features"])
stations[["properties.code", "properties.name", "properties.altitude"]]

,properties.code,properties.name,properties.altitude
0,6414,BEITEM,24.80
1,6418,ZEEBRUGGE,12.10
2,6438,STABROEK,4.00
3,6447,UCCLE,100.60
4,6439,SINT-KATELIJNE-WAVER,8.30
5,6434,MELLE,15.00
6,6459,ERNAGE,159.20
7,6464,RETIE,21.50
8,6477,DIEPENBEEK,39.30
9,10101002,DE HAAN,3.66


Uccle has code 6447.

**9.2** Was 2024 warmer or wetter in Uccle than 2023 (`../../data/raw/kmi_daily_2023.csv`)? Fetch the daily
observations of Uccle in 2024 (dataset `aws:aws_1day`). To fetch only some of the rows, add a `CQL_FILTER`
parameter. For example, this one keeps the days of June 2023 at Zeebrugge (code 6418):

```python
"CQL_FILTER": "code = 6418 AND timestamp DURING 2023-06-01T00:00:00Z/2023-06-30T23:59:59Z",
```

In [3]:
params = {
    "service": "WFS",
    "version": "2.0.0",
    "request": "GetFeature",
    "typeNames": "aws:aws_1day",
    "outputFormat": "json",
    "CQL_FILTER": "code = 6447 AND timestamp DURING 2024-01-01T00:00:00Z/2024-12-31T23:59:59Z",
}
response = requests.get(KMI_URL, params=params, timeout=60)
response.raise_for_status()
uccle_2024 = pd.json_normalize(response.json()["features"])
len(uccle_2024)

366

In [4]:
uccle_2024["properties.temp_avg"].mean()

np.float64(11.8698087431694)

In [5]:
uccle_2024["properties.precip_quantity"].sum()

np.float64(1170.72)

In [6]:
daily = pd.read_csv("../../data/raw/kmi_daily_2023.csv")
uccle_2023 = daily[daily["name"] == "UCCLE"]
uccle_2023["temp_avg"].mean()

np.float64(12.14413698630137)

In [7]:
uccle_2023["precip_quantity"].sum()

np.float64(1011.4199999999998)

2024 was a little cooler (11.9 against 12.1 °C) and clearly wetter (1 171 against 1 011 mm).

## Bonus

Explore the rest of the KMI data, and fetch something you find interesting. What the datasets and their
columns mean: https://opendata.meteo.be/documentation/?dataset=aws

While exploring, always add `"count": 100` to the parameters: it fetches only the first 100 rows. Without
it, `aws:aws_1hour` alone returns 2.6 million rows.

There is no single answer.